# 11 — HITL closed loop

Walks one correction through the full HITL loop: sample low-confidence chunk → enqueue → record correction → retrain dictionaries → verify downstream artifact updated.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'apps').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')
from apps.backend.feedback.hitl import sample_low_confidence, apply_correction
from apps.backend.graph.neo4j_client import get_driver

driver = get_driver()
sample = sample_low_confidence(driver, k=1)
print('sampled:', sample)
if sample:
    res = apply_correction(driver, chunk_id=sample[0]['chunk_id'], span='', replacement='', actor='notebook-11')
    print('correction:', res)

ART = REPO_ROOT / 'notebooks' / '_artifacts' / '11_hitl_closed_loop'
ART.mkdir(parents=True, exist_ok=True)
(ART / 'loop.json').write_text(json.dumps({'ts': datetime.now(timezone.utc).isoformat(), 'sample': sample}, ensure_ascii=False, indent=2, default=str))
print('artifact:', ART / 'loop.json')
